# BF5 — Context-Aware Recommendations + SPARQL Reasoning

Multi-signal artwork recommendation using ALL semantic signals from the B0-B6 pipeline.
Merges SPARQL reasoning (SR-30..34) with Python-side multi-signal scoring.

**Evaluation protocols:**
1. Trajectory split (70/30) — primary: first 70% as input, predict remaining 30%
2. Leave-k-out (last 5) — hide last 5, test pattern capture
3. Rank-based — NDCG/AUC on unvisited artworks ranked by score

**Signals:** semantic_sim (0.30), nav_match (0.20), significant (0.15), dominant_ctx (0.15), user_pref (0.10), engagement (0.10)

In [ ]:
import sys
import os

# Ensure 'helpers' package is importable regardless of Jupyter launch directory
for _d in [os.getcwd(),
           os.path.join(os.getcwd(), 'bird', 'notebooks'),
           os.path.join(os.getcwd(), 'notebooks')]:
    if os.path.isdir(os.path.join(_d, 'helpers')):
        if _d not in sys.path:
            sys.path.insert(0, _d)
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

from helpers.museum_recommender import (
    MuseumRecommender
)
from helpers.semantic_reasoning import SemanticReasoningEngine

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Setup + Data Loading

In [ ]:
rec = MuseumRecommender()
rec.load_all_data(features_path='output/semantic_features/trajectory_features.json')

print(f"\nTrajectories: {len(rec.sequences)}")
print(f"Artworks: {len(rec.all_artwork_uris)}")
print(f"Avg sequence length: {np.mean([len(s) for s in rec.sequences.values()]):.1f}")
print(f"Users with preferences: {len(rec.user_preferences)}")

In [ ]:
# Distribution of navigation styles and engagement levels
nav_counts = Counter()
eng_counts = Counter()
for traj, meta in rec.traj_metadata.items():
    styles = meta.get('nav_styles', set())
    if styles:
        for s in styles:
            nav_counts[s] += 1
    else:
        nav_counts['None'] += 1
    eng = meta.get('engagement')
    eng_counts[eng or 'None'] += 1

print("Navigation styles:", dict(nav_counts))
print("Engagement levels:", dict(eng_counts))

## 2. Baseline — BF4 Collaborative Filtering Anchor

BF4 achieved Hit@20 = 0.098 with user-based CF on binary feedback. Reproduce as anchor.

In [ ]:
# BF4 CF baseline (reproduced from BF4)
from pathlib import Path

cf_path = Path('output') / 'explicit_feedback_visitors.csv'
if cf_path.exists():
    cf_df = pd.read_csv(cf_path)
    print(f"CF feedback: {len(cf_df)} records, {cf_df['visitor_id'].nunique()} visitors, "
          f"{cf_df['item_id'].nunique()} items")
else:
    print("CF feedback file not found — BF4 baseline will be skipped")
    cf_df = None

In [ ]:
# Popularity baseline quick check
pop_result = rec.evaluate_recommendations(
    method='popularity', k_values=[5, 10, 20], protocol='trajectory_split'
)
print("Popularity baseline (trajectory split):")
for k in [5, 10, 20]:
    ci = pop_result[f'recall@{k}_ci']
    print(f"  Recall@{k}: {pop_result[f'recall@{k}']:.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"  MRR: {pop_result['mrr']:.4f}")

## 3. SPARQL Reasoning Pipeline

Run SR-05..SR-34 rules to materialize co-visitation, sequence, engagement, similarity, preference, and recommendation triples.

In [ ]:
sr = SemanticReasoningEngine()
sr.clear_reasoning_triples()
print("Cleared previous reasoning triples")

In [ ]:
pipeline_results = sr.run_full_pipeline(include_similarity_batch=True)

print("\nPipeline results:")
for step, result in pipeline_results.items():
    print(f"  {step}: {result}")

In [ ]:
# Verify preferences and engagement (computed at load time from interaction data)
print(f"Users with preferences: {len(rec.user_preferences)}")

eng_counts = Counter()
for traj, meta in rec.traj_metadata.items():
    eng = meta.get('engagement')
    eng_counts[eng or 'None'] += 1
print(f"Engagement levels: {dict(eng_counts)}")

# Show sample preferences
for user, prefs in list(rec.user_preferences.items())[:3]:
    top = [(p['category'], round(p['score'], 3)) for p in prefs[:3]]
    print(f"  {user.split('/')[-1]}: {top}")

In [ ]:
# Materialized triple counts
triple_counts = sr.count_materialized_triples()
print("\nMaterialized triples by predicate:")
for pred, count in sorted(triple_counts.items(), key=lambda x: -x[1]):
    print(f"  {pred}: {count}")

# Relationship stats
rel_stats = sr.get_relationship_stats()
print(f"\nCo-visitation links: {rel_stats['covisitation']}")
print(f"Sequence links: {rel_stats['sequence']}")

In [ ]:
# Rule yield per trajectory (sample 5)
traj_uris = list(rec.sequences.keys())[:5]
for traj in traj_uris:
    yields = sr.get_rule_yield(traj)
    total = sum(yields.values())
    print(f"\n{traj.split('/')[-1]}: {total} total POIs")
    for rule, count in yields.items():
        print(f"  {rule}: {count}")

## 4. Signal Analysis

Analyze the discriminative power of each signal: navigation style, engagement, significant POIs, dominant co-visitation.

In [ ]:
# Navigation style artwork patterns
nav_artwork_counts = defaultdict(Counter)
for traj, meta in rec.traj_metadata.items():
    styles = meta.get('nav_styles', set()) or {'NoStyle'}
    arts = rec.traj_artwork_sets.get(traj, set())
    for style in styles:
        for art in arts:
            cat = rec.poi_metadata.get(art, {}).get('category', 'Unknown')
            nav_artwork_counts[style][cat] += 1

# Display as heatmap
nav_df = pd.DataFrame(nav_artwork_counts).fillna(0).astype(int)
if not nav_df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(nav_df, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
    ax.set_title('Artwork Categories by Navigation Style')
    ax.set_ylabel('Category')
    plt.tight_layout()
    plt.show()

In [ ]:
# Engagement grouping — artworks per engagement level
eng_art_counts = defaultdict(lambda: {'unique_arts': set(), 'avg_seq_len': []})
for traj, meta in rec.traj_metadata.items():
    eng = meta.get('engagement', 'None')
    arts = rec.traj_artwork_sets.get(traj, set())
    eng_art_counts[eng]['unique_arts'].update(arts)
    eng_art_counts[eng]['avg_seq_len'].append(len(rec.sequences.get(traj, [])))

print("Engagement level analysis:")
for eng, data in sorted(eng_art_counts.items()):
    avg_len = np.mean(data['avg_seq_len']) if data['avg_seq_len'] else 0
    print(f"  {eng}: {len(data['unique_arts'])} unique artworks, "
          f"avg seq length {avg_len:.1f}, {len(data['avg_seq_len'])} trajectories")

In [ ]:
# Significant POI coverage
sig_arts = rec._get_significant_artworks()
print(f"Significant artworks: {len(sig_arts)}")

sig_coverage = []
for traj in rec.sequences:
    visited = rec.traj_artwork_sets.get(traj, set())
    coverage = len(visited & sig_arts) / len(sig_arts) if sig_arts else 0
    sig_coverage.append(coverage)

print(f"Mean coverage: {np.mean(sig_coverage):.3f}")
print(f"Min/Max: {np.min(sig_coverage):.3f} / {np.max(sig_coverage):.3f}")

In [ ]:
# Dominant co-visitation heatmap (top 15 artworks)
dominant_covisits = Counter()
for traj, interactions in rec.interaction_data.items():
    dominant = None
    meaningful = []
    for item in interactions:
        if item['int_type'] == 'DominantInteraction':
            dominant = item['poi']
        elif item['int_type'] in ('MeaningfulInteraction', 'ImmersiveInteraction'):
            meaningful.append(item['poi'])
    if dominant:
        for art in meaningful:
            if art != dominant:
                d_label = rec.poi_metadata.get(dominant, {}).get('label', dominant.split('/')[-1])[:20]
                a_label = rec.poi_metadata.get(art, {}).get('label', art.split('/')[-1])[:20]
                dominant_covisits[(d_label, a_label)] += 1

if dominant_covisits:
    top_dominants = Counter()
    for (d, a), c in dominant_covisits.items():
        top_dominants[d] += c
    top_d_labels = [d for d, _ in top_dominants.most_common(8)]
    top_a_labels = [a for (d, a), c in dominant_covisits.most_common(30) if d in top_d_labels][:15]

    matrix = pd.DataFrame(0, index=top_d_labels, columns=list(dict.fromkeys(top_a_labels)))
    for (d, a), c in dominant_covisits.items():
        if d in matrix.index and a in matrix.columns:
            matrix.loc[d, a] = c

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title('Dominant Artwork Co-visitation Counts')
    ax.set_ylabel('Dominant Artwork')
    ax.set_xlabel('Co-visited Artwork')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 5. Multi-Signal Recommendation

Per-signal score breakdown for 3 sample trajectories. Compare with SPARQL SR-30..34 recommendations.

In [ ]:
# Pick 3 diverse trajectories: one Exploratory, one Immersive, one NoStyle
sample_trajs = []
exploratory = immersive = nostyle = None

for traj, meta in rec.traj_metadata.items():
    styles = meta.get('nav_styles', set())
    if not exploratory and 'ExploratoryMovement' in styles:
        exploratory = traj
    elif not immersive and 'ImmersiveMovement' in styles:
        immersive = traj
    elif not nostyle and not styles:
        nostyle = traj

for t in [exploratory, immersive, nostyle]:
    if t:
        sample_trajs.append(t)

# Fill remaining if needed
all_trajs = list(rec.sequences.keys())
while len(sample_trajs) < 3 and all_trajs:
    t = all_trajs.pop(0)
    if t not in sample_trajs:
        sample_trajs.append(t)

for traj in sample_trajs:
    meta = rec.traj_metadata.get(traj, {})
    print(f"\n{'='*60}")
    print(f"Trajectory: {traj.split('/')[-1]}")
    print(f"  Nav: {meta.get('nav_styles', set())}")
    print(f"  Engagement: {meta.get('engagement')}")
    print(f"  Sequence length: {len(rec.sequences.get(traj, []))}")
    print(f"  Artworks visited: {len(rec.traj_artwork_sets.get(traj, set()))}")

In [ ]:
# Multi-signal recommendations for sample trajectories
for traj in sample_trajs:
    recs = rec.context_aware_recommend(traj, top_k=5)
    print(f"\n--- {traj.split('/')[-1]} top-5 ---")
    for i, r in enumerate(recs, 1):
        signals_str = ', '.join(f"{k}={v:.3f}" for k, v in sorted(r['signals'].items()) if v > 0)
        print(f"  {i}. {r['label'][:40]} ({r['category']}, floor {r['floor']}) "
              f"score={r['score']:.4f} [{signals_str}]")

In [ ]:
# SPARQL SR-30..34 recommendations for same trajectories
for traj in sample_trajs:
    sparql_recs = sr.recommend_with_explanations(traj, top_k=5)
    print(f"\n--- {traj.split('/')[-1]} SPARQL top-5 ---")
    for i, r in enumerate(sparql_recs, 1):
        print(f"  {i}. {r['poiLabel'][:40]} score={r['score']:.3f} "
              f"rules={r['ruleCoverage']} {r['rules']}")

In [ ]:
# Overlap analysis: multi-signal vs SPARQL
print("\nOverlap analysis (top-20):")
for traj in sample_trajs:
    ms_recs = rec.context_aware_recommend(traj, top_k=20)
    sr_recs = sr.recommend(traj, top_k=20)
    ms_pois = set(r['poi'] for r in ms_recs)
    sr_pois = set(r['poi'] for r in sr_recs)
    overlap = ms_pois & sr_pois
    print(f"  {traj.split('/')[-1]}: multi-signal={len(ms_pois)}, SPARQL={len(sr_pois)}, "
          f"overlap={len(overlap)} ({len(overlap)/max(len(ms_pois|sr_pois),1)*100:.0f}%)")

## 6. Ablation Study

Evaluate with each signal turned off: [full, -semantic, -nav, -significant, -dominant, -pref, -engagement, baselines]

In [ ]:
print("Running ablation study (trajectory split protocol)...")
ablation = rec.ablation_study(k_values=[5, 10, 20], protocol='trajectory_split')

In [ ]:
# Format ablation results table
rows = []
for config, result in ablation.items():
    row = {'Config': config, 'N': result.get('n_evaluated', 0)}
    for k in [5, 10, 20]:
        row[f'Recall@{k}'] = f"{result.get(f'recall@{k}', 0):.4f}"
        ci = result.get(f'recall@{k}_ci', (0, 0))
        row[f'R@{k} CI'] = f"[{ci[0]:.4f}, {ci[1]:.4f}]"
    row['MRR'] = f"{result.get('mrr', 0):.4f}"
    rows.append(row)

ablation_df = pd.DataFrame(rows)
display(ablation_df)

# Extended metrics table
print("\n=== Extended Metrics ===")
ext_rows = []
for config, result in ablation.items():
    row = {'Config': config}
    for k in [5, 10, 20]:
        row[f'P@{k}'] = f"{result.get(f'precision@{k}', 0):.4f}"
        row[f'HR@{k}'] = f"{result.get(f'hit_rate@{k}', 0):.3f}"
        row[f'Cov@{k}'] = f"{result.get(f'coverage@{k}', 0):.3f}"
    row['MAP'] = f"{result.get('map', 0):.4f}"
    row['AvgCand'] = f"{result.get('avg_candidates', 0):.0f}"
    ext_rows.append(row)

ext_df = pd.DataFrame(ext_rows)
display(ext_df)

In [ ]:
# Ablation bar chart
configs = list(ablation.keys())
recall20_vals = [ablation[c].get('recall@20', 0) for c in configs]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(configs)), recall20_vals, color=['#2196F3' if c == 'full' else
              '#FF9800' if c.startswith('-') else '#4CAF50' for c in configs])
ax.set_xticks(range(len(configs)))
ax.set_xticklabels(configs, rotation=45, ha='right')
ax.set_ylabel('Recall@20')
ax.set_title('Ablation Study — Recall@20 by Configuration')
ax.axhline(y=recall20_vals[0], color='blue', linestyle='--', alpha=0.3, label='full model')

for bar, val in zip(bars, recall20_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 6c. Split Ratio Sensitivity

The 70/30 trajectory split is arbitrary. Sweeping ratios 0.10-0.90 reveals:
- How early in a visit the system provides value (low ratios = few observed artworks)
- How the directional filter interacts with input length
- The optimal operating point for each configuration

In [ ]:
print("Running split ratio sweep...")
sweep_results = rec.sweep_split_ratios()

# Build summary table
rows = []
for ratio, configs in sorted(sweep_results.items()):
    for config, metrics in configs.items():
        rows.append({
            'ratio': ratio,
            'config': config,
            'recall@5': metrics.get('recall@5', 0),
            'recall@10': metrics.get('recall@10', 0),
            'recall@20': metrics.get('recall@20', 0),
            'mrr': metrics.get('mrr', 0),
            'avg_candidates': metrics.get('avg_candidates', 0),
        })

sweep_df = pd.DataFrame(rows)
pivot = sweep_df.pivot_table(index='ratio', columns='config', values='recall@20')
print("\nRecall@20 by split ratio and configuration:")
print(pivot.round(4).to_string())

mrr_pivot = sweep_df.pivot_table(index='ratio', columns='config', values='mrr')
print("\nMRR by split ratio and configuration:")
print(mrr_pivot.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs_to_plot = [c for c in sweep_df['config'].unique()]
colors = {'full': '#2196F3', 'hybrid': '#FF9800', 'flow_only': '#4CAF50', 'popularity': '#9E9E9E'}

for config in configs_to_plot:
    cfg_data = sweep_df[sweep_df['config'] == config].sort_values('ratio')
    color = colors.get(config, '#333')
    axes[0].plot(cfg_data['ratio'], cfg_data['recall@20'],
                 marker='o', label=config, color=color, linewidth=2, markersize=5)
    axes[1].plot(cfg_data['ratio'], cfg_data['mrr'],
                 marker='s', label=config, color=color, linewidth=2, markersize=5)

axes[0].set_xlabel('Split Ratio (input fraction)')
axes[0].set_ylabel('Recall@20')
axes[0].set_title('Recall@20 vs Split Ratio')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Split Ratio (input fraction)')
axes[1].set_ylabel('MRR')
axes[1].set_title('MRR vs Split Ratio')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Candidate set size vs ratio (shows directional filter narrowing)
full_data = sweep_df[sweep_df['config'] == 'full'].sort_values('ratio')

if 'avg_candidates' in full_data.columns and full_data['avg_candidates'].sum() > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(full_data['ratio'], full_data['avg_candidates'],
            marker='D', color='#2196F3', linewidth=2, markersize=6)
    ax.set_xlabel('Split Ratio (input fraction)')
    ax.set_ylabel('Avg Forward Candidates')
    ax.set_title('Directional Filter: Candidate Set Size vs Input Length')
    ax.grid(True, alpha=0.3)

    # Annotate holdout size on secondary axis
    ax2 = ax.twinx()
    avg_seq = np.mean([len(s) for s in rec.sequences.values()])
    holdout_sizes = [avg_seq * (1 - r) for r in full_data['ratio']]
    ax2.plot(full_data['ratio'], holdout_sizes,
             marker='x', color='#F44336', linewidth=1.5, linestyle='--', alpha=0.7)
    ax2.set_ylabel('Avg Holdout Size', color='#F44336')
    ax2.tick_params(axis='y', labelcolor='#F44336')

    plt.tight_layout()
    plt.show()
else:
    print("No candidate size data (popularity baseline doesn't track candidates)")

## 6b. Hybrid Recommendation — Context + Flow

Two strategies for combining context-aware (6 signals) with flow prediction:
- **Strategy A (Integrated):** `flow_prob` as 7th signal in the precompute/cache framework. Weight sweep from 10-50%.
- **Strategy B (Simple merge):** Independent ranked lists from `context_aware_recommend` and `predict_next_artwork`, normalized and merged.

In [ ]:
# Strategy A: Integrated hybrid — ablation includes hybrid, flow_only, and weight sweep configs
# (Already included in ablation_study via the updated configs)
# Extract hybrid-related results from ablation
hybrid_configs = {k: v for k, v in ablation.items()
                  if k in ('hybrid', 'flow_only') or k.startswith('flow_')}

if hybrid_configs:
    print('=== Strategy A: Integrated Hybrid (flow as 7th signal) ===\n')
    rows = []
    for config, result in hybrid_configs.items():
        rows.append({
            'Config': config,
            'Recall@5': f"{result.get('recall@5', 0):.3f}",
            'Recall@10': f"{result.get('recall@10', 0):.3f}",
            'Recall@20': f"{result.get('recall@20', 0):.3f}",
            'MRR': f"{result.get('mrr', 0):.3f}",
            'N': result.get('n_evaluated', 0),
        })
    df_hybrid = pd.DataFrame(rows)
    display(df_hybrid)
else:
    print('No hybrid configs found — re-run ablation with updated museum_recommender.py')

In [ ]:
# Strategy B: Simple merge evaluation
print('=== Strategy B: Simple Merge (context 0.5 + flow 0.5) ===')
merge_result = rec.evaluate_recommendations(
    method='hybrid_merge', k_values=[5, 10, 20], protocol='trajectory_split'
)
print(f"Recall@5:  {merge_result['recall@5']:.3f}  {merge_result.get('recall@5_ci', '')}")
print(f"Recall@10: {merge_result['recall@10']:.3f}  {merge_result.get('recall@10_ci', '')}")
print(f"Recall@20: {merge_result['recall@20']:.3f}  {merge_result.get('recall@20_ci', '')}")
print(f"MRR:       {merge_result['mrr']:.3f}  {merge_result.get('mrr_ci', '')}")
print(f"N evaluated: {merge_result['n_evaluated']}")

In [ ]:
# Comparison table: full (context-only), hybrid (integrated), hybrid_merge (simple), flow_only, popularity
comparison_configs = {
    'full (context-only)': ablation.get('full', {}),
    'hybrid (integrated)': ablation.get('hybrid', {}),
    'hybrid_merge (simple)': merge_result,
    'flow_only': ablation.get('flow_only', {}),
    'popularity': ablation.get('popularity', {}),
}

rows = []
for name, result in comparison_configs.items():
    if not result:
        continue
    rows.append({
        'Method': name,
        'Recall@5': f"{result.get('recall@5', 0):.3f}",
        'Recall@10': f"{result.get('recall@10', 0):.3f}",
        'Recall@20': f"{result.get('recall@20', 0):.3f}",
        'P@20': f"{result.get('precision@20', 0):.3f}",
        'HR@20': f"{result.get('hit_rate@20', 0):.3f}",
        'MRR': f"{result.get('mrr', 0):.3f}",
        'MAP': f"{result.get('map', 0):.3f}",
        'Cov@20': f"{result.get('coverage@20', 0):.3f}",
    })

df_compare = pd.DataFrame(rows)
print('=== Hybrid Recommendation Comparison ===')
display(df_compare)

In [ ]:
# Weight sweep plot: Recall@20 and MRR vs flow_prob weight
sweep_configs = {k: v for k, v in ablation.items() if k.startswith('flow_') and 'pct' in k}
sweep_configs['full (0%)'] = ablation.get('full', {})

if sweep_configs:
    flow_pcts = []
    recall20_vals = []
    mrr_vals = []
    for name in sorted(sweep_configs.keys()):
        result = sweep_configs[name]
        if 'pct' in name:
            pct = int(name.split('_')[1].replace('pct', ''))
        else:
            pct = 0
        flow_pcts.append(pct)
        recall20_vals.append(result.get('recall@20', 0))
        mrr_vals.append(result.get('mrr', 0))

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    ax1.plot(flow_pcts, recall20_vals, 'o-', color='#2196F3', linewidth=2, label='Recall@20')
    ax2.plot(flow_pcts, mrr_vals, 's--', color='#FF9800', linewidth=2, label='MRR')

    ax1.set_xlabel('Flow Weight (%)')
    ax1.set_ylabel('Recall@20', color='#2196F3')
    ax2.set_ylabel('MRR', color='#FF9800')
    ax1.set_title('Weight Sweep: Flow Prediction Signal Weight vs Metrics')
    ax1.set_xticks(flow_pcts)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

    plt.tight_layout()
    plt.show()

    # Identify peak
    best_idx = recall20_vals.index(max(recall20_vals))
    print(f'Peak Recall@20 = {recall20_vals[best_idx]:.3f} at flow_weight = {flow_pcts[best_idx]}%')
else:
    print('No sweep configs found in ablation results')

## 7. Evaluation — Three Protocols

### 7a. Trajectory Split (70/30) — Primary

In [ ]:
print("=== Trajectory Split (70/30) ===")
ts_result = rec.evaluate_recommendations(
    method='context_aware', k_values=[5, 10, 20], protocol='trajectory_split'
)
for k in [5, 10, 20]:
    ci = ts_result[f'recall@{k}_ci']
    print(f"Recall@{k}: {ts_result[f'recall@{k}']:.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"MRR: {ts_result['mrr']:.4f} {ts_result['mrr_ci']}")
print(f"MAP: {ts_result['map']:.4f} {ts_result['map_ci']}")
print(f"Evaluated: {ts_result['n_evaluated']} trajectories")

print("\nPrecision and Hit Rate:")
for k in [5, 10, 20]:
    p = ts_result.get(f'precision@{k}', 0)
    hr = ts_result.get(f'hit_rate@{k}', 0)
    cov = ts_result.get(f'coverage@{k}', 0)
    print(f"  @{k}: P={p:.4f}, HR={hr:.3f}, Coverage={cov:.3f}")

print("\nPer navigation style:")
for nav, metrics in ts_result['nav_style_breakdown'].items():
    r20 = metrics.get('recall@20', 0)
    print(f"  {nav} (n={metrics.get('n', 0)}): Recall@20={r20:.4f}")

### 7b. Leave-k-out (last 5)

In [ ]:
print("=== Leave-5-out ===")
lko_result = rec.evaluate_recommendations(
    method='context_aware', k_values=[5, 10, 20], protocol='leave_k_out', leave_k=5
)
for k in [5, 10, 20]:
    ci = lko_result[f'recall@{k}_ci']
    print(f"Recall@{k}: {lko_result[f'recall@{k}']:.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"MRR: {lko_result['mrr']:.4f} {lko_result['mrr_ci']}")
print(f"MAP: {lko_result['map']:.4f} {lko_result['map_ci']}")
print(f"Evaluated: {lko_result['n_evaluated']} trajectories")

print("\nPrecision and Hit Rate:")
for k in [5, 10, 20]:
    p = lko_result.get(f'precision@{k}', 0)
    hr = lko_result.get(f'hit_rate@{k}', 0)
    cov = lko_result.get(f'coverage@{k}', 0)
    print(f"  @{k}: P={p:.4f}, HR={hr:.3f}, Coverage={cov:.3f}")

print("\nPer navigation style:")
for nav, metrics in lko_result['nav_style_breakdown'].items():
    r20 = metrics.get('recall@20', 0)
    print(f"  {nav} (n={metrics.get('n', 0)}): Recall@20={r20:.4f}")

### 7c. Rank-based (NDCG, AUC)

In [ ]:
print("=== Rank-based ===")
rank_result = rec.evaluate_recommendations_rank_based(k_values=[5, 10, 20])
for k in [5, 10, 20]:
    ci = rank_result.get(f'ndcg@{k}_ci', (0, 0))
    print(f"NDCG@{k}: {rank_result.get(f'ndcg@{k}', 0):.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
ci_auc = rank_result.get('auc_ci', (0, 0))
print(f"AUC: {rank_result.get('auc', 0):.4f} [{ci_auc[0]:.4f}, {ci_auc[1]:.4f}]")
print(f"Evaluated: {rank_result.get('n_evaluated', 0)} trajectories")

In [ ]:
# Comparison table across protocols
comparison_rows = [
    {'Protocol': 'Trajectory Split (70/30)',
     'Recall@5': f"{ts_result['recall@5']:.4f}",
     'Recall@10': f"{ts_result['recall@10']:.4f}",
     'Recall@20': f"{ts_result['recall@20']:.4f}",
     'P@20': f"{ts_result.get('precision@20', 0):.4f}",
     'HR@20': f"{ts_result.get('hit_rate@20', 0):.3f}",
     'MRR': f"{ts_result['mrr']:.4f}",
     'MAP': f"{ts_result.get('map', 0):.4f}"},
    {'Protocol': 'Leave-5-out',
     'Recall@5': f"{lko_result['recall@5']:.4f}",
     'Recall@10': f"{lko_result['recall@10']:.4f}",
     'Recall@20': f"{lko_result['recall@20']:.4f}",
     'P@20': f"{lko_result.get('precision@20', 0):.4f}",
     'HR@20': f"{lko_result.get('hit_rate@20', 0):.3f}",
     'MRR': f"{lko_result['mrr']:.4f}",
     'MAP': f"{lko_result.get('map', 0):.4f}"},
    {'Protocol': 'Rank-based',
     'Recall@5': '—',
     'Recall@10': '—',
     'Recall@20': '—',
     'P@20': '—',
     'HR@20': '—',
     'MRR': f"NDCG@20={rank_result.get('ndcg@20', 0):.4f}",
     'MAP': f"AUC={rank_result.get('auc', 0):.4f}"},
]
display(pd.DataFrame(comparison_rows))

In [ ]:
# Baselines already computed in ablation study — extract and display
print("=== Baseline comparison (from ablation) ===")
for method in ['popularity', 'random', 'same_floor']:
    result = ablation.get(method, {})
    if result:
        print(f"\n{method} (n={result.get('n_evaluated', 0)}):")
        for k in [5, 10, 20]:
            ci = result.get(f'recall@{k}_ci', (0, 0))
            print(f"  Recall@{k}: {result.get(f'recall@{k}', 0):.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
        print(f"  MRR: {result.get('mrr', 0):.4f}")

# Context-aware (full model) for comparison
full = ablation.get('full', {})
print(f"\ncontext_aware (n={full.get('n_evaluated', 0)}):")
for k in [5, 10, 20]:
    ci = full.get(f'recall@{k}_ci', (0, 0))
    print(f"  Recall@{k}: {full.get(f'recall@{k}', 0):.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"  MRR: {full.get('mrr', 0):.4f}")

## 8. Qualitative Examples

3 diverse trajectories: visitor profile, top-5 recommendations with explanations, which signals contributed most.

In [ ]:
for traj in sample_trajs:
    meta = rec.traj_metadata.get(traj, {})
    user = rec.traj_to_user.get(traj, 'Unknown')
    seq = rec.sequences.get(traj, [])
    dominant = rec._get_dominant_artwork(traj)

    print(f"\n{'='*70}")
    print(f"Trajectory: {traj.split('/')[-1]}")
    print(f"User: {user.split('/')[-1]}")
    print(f"Navigation: {meta.get('nav_styles', set())}")
    print(f"Engagement: {meta.get('engagement')}")
    print(f"Sequence: {len(seq)} artworks")
    if dominant:
        d_label = rec.poi_metadata.get(dominant, {}).get('label', '?')
        print(f"Dominant: {d_label}")

    # Category breakdown
    cats = Counter(rec.poi_metadata.get(a, {}).get('category', '?') for a in seq)
    print(f"Categories: {dict(cats)}")

    # User preferences
    prefs = rec.user_preferences.get(user, [])
    if prefs:
        print(f"Preferences: {[(p['category'], round(p['score'], 2)) for p in prefs[:5]]}")

    # Top-5 recommendations with explainability
    recs = rec.context_aware_recommend(traj, top_k=5, explain=True)
    print("\nTop-5 Recommendations:")
    for i, r in enumerate(recs, 1):
        print(f"  {i}. {r['label'][:45]} ({r['category']}, floor {r['floor']})")
        print(f"     Score: {r['score']:.4f}")
        for exp in r.get('explanations', [])[:3]:
            print(f"     [{exp['signal']}] contribution={exp['contribution']:.4f}")
            print(f"       {exp['evidence']}")
            print(f"       KG: {exp['kg_predicate']}")
            print(f"       Pipeline: {exp['pipeline_step']}")

## 9. Recommendation Maps

Spatial visualization of recommendations on the museum floor plan. Blue numbered path = visitor's actual trajectory. Red stars = recommended artworks. Gray dots = all artworks for spatial context.

In [ ]:
# Recommendation maps for the 3 sample trajectories
for traj in sample_trajs:
    if not rec.position_coords.get(traj):
        print(f"No coordinate data for {traj.split('/')[-1]}, skipping map")
        continue

    fig = rec.plot_recommendation_map(traj, top_k=10)
    plt.show()
    print()

In [ ]:
# Side-by-side: multi-signal vs SPARQL recommendations on the floor plan
# For the first sample trajectory, show both recommendation sets spatially
traj = sample_trajs[0]
if traj in rec.position_coords and rec.position_coords[traj]:
    ms_recs = rec.context_aware_recommend(traj, top_k=10)
    sr_recs_raw = sr.recommend(traj, top_k=10)

    ms_pois = [r['poi'] for r in ms_recs]
    sr_pois = [r['poi'] for r in sr_recs_raw]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))

    rec.plot_trajectory_on_floor(traj, predicted_pois=ms_pois, ax=ax1)
    ax1.set_title(f'Multi-Signal Recommendations — {traj.split("/")[-1]}', fontsize=10)

    rec.plot_trajectory_on_floor(traj, predicted_pois=sr_pois, ax=ax2)
    ax2.set_title(f'SPARQL Rules Recommendations — {traj.split("/")[-1]}', fontsize=10)

    plt.tight_layout()
    plt.show()

    overlap = set(ms_pois) & set(sr_pois)
    print(f"Overlap: {len(overlap)} artworks in common out of top-10")
    print(f"Multi-signal only: {len(set(ms_pois) - set(sr_pois))}")
    print(f"SPARQL only: {len(set(sr_pois) - set(ms_pois))}")

In [ ]:
# Per-floor recommendation maps for the first trajectory
# Useful when the visitor spans multiple floors
traj = sample_trajs[0]
if traj in rec.position_coords and rec.position_coords[traj]:
    visited_floors = sorted(set(c['floor'] for c in rec.position_coords[traj]))

    if len(visited_floors) > 1:
        recs = rec.context_aware_recommend(traj, top_k=10)
        rec_pois = [r['poi'] for r in recs]

        fig, axes = plt.subplots(1, len(visited_floors),
                                 figsize=(7 * len(visited_floors), 8))
        if len(visited_floors) == 1:
            axes = [axes]

        for i, floor in enumerate(visited_floors):
            rec.plot_trajectory_on_floor(
                traj, predicted_pois=rec_pois, floor_filter=floor, ax=axes[i]
            )

        plt.suptitle(f'Per-Floor Recommendations — {traj.split("/")[-1]}',
                     fontsize=12, y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Trajectory visits only floor {visited_floors[0]}, single-floor view already shown above")